In [1]:
import os
from dotenv import load_dotenv
from langchain_groq import ChatGroq
from langchain_huggingface import HuggingFaceEmbeddings

load_dotenv()

db_params = {
    "dbname": "data-sense-db",
    "user": os.getenv("POSTGRES_USERNAME"),
    "password": os.getenv("POSTGRES_PASSWORD"),
    "host": os.getenv("POSTGRES_HOST"),
    "port": "5432",
}
os.environ["HF_TOKEN"] = os.getenv("HF_TOKEN")
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

llm = ChatGroq(
    model="llama-3.3-70b-versatile", api_key=os.getenv("GROQ_API_KEY"), temperature=0
)

llm.invoke("Hello")

d:\ai-learning\data-sense\agent-service\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 8845.30it/s]


AIMessage(content='Hello. How can I help you today?', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 10, 'prompt_tokens': 36, 'total_tokens': 46, 'completion_time': 0.026342187, 'completion_tokens_details': None, 'prompt_time': 0.001697116, 'prompt_tokens_details': None, 'queue_time': 0.057940998, 'total_time': 0.028039303}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_dae98b5ecb', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019e68e7-0990-7e82-845d-5b2f0d246624-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 36, 'output_tokens': 10, 'total_tokens': 46})

In [1]:
from typing import TypedDict, Optional


class AgentState(TypedDict):
    question: str
    conversation_history: list[dict]

    # Ambiguity Node
    is_ambiguous: bool
    clarifying_question: str

    # RAG - schema context
    schema_context: str

    # SQL Lifecycle
    generated_sql: str
    sql_explaination: str
    is_sql_safe: bool
    safety_error: str
    sql_result: Optional[list[dict]]
    sql_error: Optional[str]
    error_count: int

    # Output
    insight: str
    viz_code: str
    followup_questions: list[str]

    # Final Response
    response: str

## PROMPTs


#### System Prompts


In [2]:
ambiguity = """
Is this question specific enough to generate a SQL query ?
Consider the available tables listed below.

Available tables : {tables}
Conversation History:  {conversation_history}

If clear: return exactly -> CLEAR
If ambiguous: return exactly -> AMBIGUOUS: <one short clarifying question>
Nothing else. No explanation.
"""

sql_generator = """
You are an expert PostgreSQL Engineer.
Given the schema context below, write ONE optimized SQL Query
that answers the user's question.

Return ONLY the SQL - no markdown fences, no explaination.

SCHEMA : {schema_context}
CONVERSATION HISTORY (for follow-up context) : {conversation_history}

Rules:
- Never SELECT *
- Always alias aggregations: SUM(x) AS total_x
- Use LIMIT 100 unless user asks for everything
- Never use INSERT / UPDATE / DELETE / TRUNCATE / DROP
- Use date_trunc for time grouping
"""

sql_explainer = """
Explain what this SQL query does in 2-3 plain English sentences.
No technical jargon. Write as if explaining to a business user.
"""

sql_rewriter = """
You are a PostgreSQL expert debugging a failed query.
Fix the query and return ONLY the corrected SQL. No explanation. No Markdown.

SCHEMA : {schema_context}
FAILED SQL : {failed_sql}
EXACT DATABASE ERROR : {db_error}
"""

insight = """
You are a business analyst.
Summarise the query results in 2-3 plain English sentences.
Be specific - mention actual numbers and trends from the data.
If the price or cost part is involved in the insights, the currency should be INR.
"""

viz_code_generator = """
Write a Matplotlib Python snippet to visualise the data. Assume the data is already in a pandas DataFrame called `df`.
RULES: 
- No ```python fences
- No ``` backticks of any kind
- No explanation or comments
- First line must start with 'import' or 'fig'

FIGURE SIZE RULES — calculate dynamically based on data:
- For bar charts: width = max(10, len(df) * 0.8), height = 6
- For horizontal bar charts: width = 10, height = max(6, len(df) * 0.5)
- For line charts: width = max(10, len(df) * 0.4), height = 6
- If number of rows > 15, ALWAYS prefer horizontal bar chart (barh) over vertical bar
- If any label length > 10 characters, ALWAYS use horizontal bar chart (barh)

LABEL RULES:
- For vertical bar: rotate x labels using EXACTLY these two lines:
    ax.tick_params(axis='x', rotation=45)
    plt.setp(ax.get_xticklabels(), ha='right')
- For horizontal bar: no rotation needed, labels are on y axis
- Always use ax.set_xlabel() and ax.set_ylabel()
- Always set a title with ax.set_title(fontsize=14)
- Add value annotations on each bar:
    - Vertical bar: ax.bar_label(ax.containers[0], fmt='%.1f', padding=3)
    - Horizontal bar: ax.bar_label(ax.containers[0], fmt='%.1f', padding=3)

SPACING RULES:
- Always end with plt.tight_layout(pad=2.0)
- Always end with plt.show()
- Never use fig.show()
"""

followup_question = """
Based on the question asked and the insight returned, suggest exactly 3 short follow-up questions the user might want to ask next.
Return as a Python list of strings ONLY. No explanation. No markdown.
Example format: ["Question 1?", "Question 2?", "Question 3?"]
"""

#### Prompt Templates


In [3]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.messages import SystemMessage, HumanMessage

AMBIGUITY_PROMPT = ChatPromptTemplate.from_messages(
    [SystemMessage(content=ambiguity), HumanMessage(content="Question: {question}")]
)

SQL_PROMPT = ChatPromptTemplate.from_messages(
    [SystemMessage(content=sql_generator), HumanMessage(content="Question: {question}")]
)

SQL_EXPLAINER_PROMPT = ChatPromptTemplate.from_messages(
    [SystemMessage(content=sql_explainer), HumanMessage(content="SQL : {sql}")]
)

SQL_REWRITER_PROMPT = ChatPromptTemplate.from_messages(
    [
        SystemMessage(content=sql_rewriter),
        HumanMessage(content="Original Question : {question}"),
    ]
)

INSIGHT_WRITER_PROMPT = ChatPromptTemplate.from_messages(
    [
        SystemMessage(content=insight),
        HumanMessage(content="Question: {question}\nResults: {sql_results}"),
    ]
)

VIZ_CODE_PROMPT = ChatPromptTemplate.from_messages(
    [
        SystemMessage(content=viz_code_generator),
        HumanMessage(
            content="""Question: {question}
                Columns: {columns}
                Num rows: {num_rows}
                Sample data: {sample_data}
                Longest label: {longest_label}"""
        ),
    ]
)

FOLLOWUP_PROMPT = ChatPromptTemplate.from_messages([
    SystemMessage(content=followup_question),
    HumanMessage(content="Question: {question}\nInsight: {insight}\nColumns in result: {columns}")
])

d:\ai-learning\data-sense\agent-service\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
